# Sex-Specific Age Diagnostics

Two Mann-Whitney U analyses on raw patient data:

1. **Case-vs-control age gap within each sex** 
2. **Male-vs-female age comparison, within cases and within controls separately** 

In [1]:
import pandas as pd

BASE = "./"
HORIZONS = [1, 2, 3, 4, 5]
USYD_M = '#1A345E'   # navy
USYD_F = '#E64626'   # red-orange

df = pd.read_csv(BASE + 'modeling_dataset.csv')
repeated_cv = pd.read_csv(BASE + 'sex_specific_repeated_cv_summary.csv')
print(f'Loaded modeling_dataset.csv: {len(df)} patients ({(df.sex=="M").sum()} M, {(df.sex=="F").sum()} F)')

Loaded modeling_dataset.csv: 3050 patients (1700 M, 1350 F)


## Is the case/control age gap itself statistically significant?

Mann-Whitney U test comparing the age distribution of cases against controls, within each sex separately, at every horizon. 

In [2]:
from scipy.stats import mannwhitneyu as mannwhitneyu_test

case_control_sig_rows = []
for h in HORIZONS:
    elig = df[f'eligible_{h}y']
    lab = df[f'label_{h}y']
    for sex in ['M', 'F']:
        mask = (df['sex'] == sex) & elig
        case_age = df.loc[mask & (lab == 1), 'age']
        control_age = df.loc[mask & (lab == 0), 'age']
        stat, p = mannwhitneyu_test(case_age, control_age, alternative='two-sided')
        case_control_sig_rows.append(dict(horizon=f'{h}y', sex=sex, n_cases=len(case_age), n_controls=len(control_age),
                                           case_mean_age=case_age.mean(), control_mean_age=control_age.mean(),
                                           gap=case_age.mean() - control_age.mean(),
                                           U_statistic=stat, p_value=p))

case_control_sig_df = pd.DataFrame(case_control_sig_rows)
print('=== Mann-Whitney U: case age vs. control age, within each sex ===')
print(case_control_sig_df.round(4).to_string(index=False))
print()
print(f"Significant at p<0.05: {(case_control_sig_df.p_value < 0.05).sum()}/{len(case_control_sig_df)} sex/horizon combinations")
case_control_sig_df.to_csv(BASE + 'sex_raw_diagnostics_case_control_mannwhitney.csv', index=False)

=== Mann-Whitney U: case age vs. control age, within each sex ===
horizon sex  n_cases  n_controls  case_mean_age  control_mean_age     gap  U_statistic  p_value
     1y   M      154        1348        70.3377           64.4970  5.8406     132521.5   0.0000
     1y   F      107        1077        72.8598           61.3073 11.5525      83051.5   0.0000
     2y   M      193        1112        69.9275           64.5531  5.3744     135057.5   0.0000
     2y   F      135         849        72.1704           61.0624 11.1079      82224.0   0.0000
     3y   M      228         823        69.9781           64.6902  5.2879     118352.5   0.0000
     3y   F      152         642        71.4276           61.4766  9.9510      67830.0   0.0000
     4y   M      246         593        70.2236           64.8600  5.3635      91904.5   0.0000
     4y   F      165         455        71.4000           61.7253  9.6747      51552.5   0.0000
     5y   M      258         310        70.2829           67.0419  3.2

## Male vs. female age comparison, within cases and within controls

Test within a sex, do cases and controls differ in age

In [3]:
from scipy.stats import mannwhitneyu as mannwhitneyu_test2, chi2_contingency

mvf_mannwhitney_rows = []
chi2_rows = []

for h in HORIZONS:
    elig = df[f'eligible_{h}y']; lab = df[f'label_{h}y']
    sub = df[elig].copy(); sub['label'] = lab[elig].astype(int)

    for group_name, group_mask in [('cases', sub.label == 1), ('controls', sub.label == 0), ('all', sub.label.notna())]:
        ageM = sub.loc[group_mask & (sub.sex == 'M'), 'age']
        ageF = sub.loc[group_mask & (sub.sex == 'F'), 'age']
        stat, p = mannwhitneyu_test2(ageF, ageM, alternative='two-sided')
        mvf_mannwhitney_rows.append(dict(horizon=f'{h}y', group=group_name, n_M=len(ageM), n_F=len(ageF),
                                          median_age_M=ageM.median(), median_age_F=ageF.median(),
                                          U_statistic=stat, p_value=p))

    contingency = pd.crosstab(sub['sex'], sub['label'])
    chi2_stat, chi2_p, dof, expected = chi2_contingency(contingency)
    chi2_rows.append(dict(horizon=f'{h}y', chi2=chi2_stat, dof=dof, p_value=chi2_p,
                           n_M=contingency.loc['M'].sum() if 'M' in contingency.index else 0,
                           n_F=contingency.loc['F'].sum() if 'F' in contingency.index else 0))

mvf_mw_df = pd.DataFrame(mvf_mannwhitney_rows)
chi2_df = pd.DataFrame(chi2_rows)
print('=== Mann-Whitney U: age distribution, F vs M (within cases / within controls / all) ===')
print(mvf_mw_df.to_string(index=False))
print()
print('=== Chi-square test of independence: sex vs. event ===')
print(chi2_df.to_string(index=False))
mvf_mw_df.to_csv(BASE + 'sex_raw_diagnostics_mvf_mannwhitney_age.csv', index=False)
chi2_df.to_csv(BASE + 'sex_raw_diagnostics_chisquare_sex_event.csv', index=False)

=== Mann-Whitney U: age distribution, F vs M (within cases / within controls / all) ===
horizon    group  n_M  n_F  median_age_M  median_age_F  U_statistic      p_value
     1y    cases  154  107          70.0          72.0       9092.0 1.550485e-01
     1y controls 1348 1077          65.0          61.0     618788.0 3.996589e-10
     1y      all 1502 1184          65.0          62.0     775403.0 1.172354e-08
     2y    cases  193  135          70.0          71.0      14216.5 1.594729e-01
     2y controls 1112  849          65.0          61.0     395140.0 5.940693e-10
     2y      all 1305  984          66.0          62.0     558125.5 8.161390e-08
     3y    cases  228  152          70.0          71.0      18236.5 3.864779e-01
     3y controls  823  642          65.0          61.0     226197.0 2.250134e-06
     3y      all 1051  794          66.0          63.0     370126.5 3.180840e-05
     4y    cases  246  165          70.0          71.0      21112.5 4.886858e-01
     4y controls  593